# Natoe Radiology Edit — dictation → structured report

Turn a radiologist's telegraphic **dictation** into a complete structured
report by **editing a supplied normal template**.

| | |
|---|---|
| train | 636 rows, 9 cols (8 inputs + `report`) |
| test | 132 rows, 8 cols |
| metric | `RES_case = 0.65·F + 0.35·I`, lower is better |

## The one idea behind the design

RES is a **template-edit** metric, not a semantic-similarity metric. It
compares each FINDINGS field of the submission against the same field of the
reference, and weights fields that had to change **3×** more than fields that
stayed normal. So the winning move is: *change what the dictation requires,
copy everything else byte-for-byte.*

That argues against asking a model to "write the report". Instead:

```
 retrieve few-shot  ->  LLM emits a JSON *patch*  ->  guard  ->  render
  (house style)        (only the fields that         (drop        (Python rebuilds
                        change + IMPRESSION)          invented      in the template's
                                                       measurements)   field order)
```

**The model never writes the report. It writes a patch. Python renders it.**
Field order, label spelling and untouched-field preservation then become
structurally impossible to get wrong, not merely unlikely.

## 1. Setup — secrets

API keys come from the environment, **never** from this notebook.

In [ ]:
import os, sys, json, re, time, statistics
from pathlib import Path

HERE = Path.cwd()
DATA = HERE / "data" if (HERE / "data").exists() else HERE

# --- secrets: local .env, or a Kaggle secret --------------------------
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# On Kaggle, add the key as a Secret and inject it instead:
#   from kaggle_secrets import UserSecretsClient
#   os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

for k in ("GROQ_API_KEY", "OPENROUTER_API_KEY"):
    print(f"{k:20s} {'set' if os.getenv(k) else '-- not set --'}")
print("data dir:", DATA)
print("csvs:", sorted(p.name for p in DATA.glob('*.csv')))

## 2. The pipeline code

Four modules, embedded here in full so the notebook stands alone. Running this
section rewrites them onto disk and imports them.

| module | role |
|---|---|
| `res_scorer.py` | local re-implementation of the RES metric |
| `pipeline.py` | retriever, prompt, JSON patch, guard, renderer |
| `config.py` | paths, provider table, defaults |
| `evaluate.py` | dev split, scoring, error analysis |

In [ ]:
%%writefile src/natoe/res_scorer.py
"""
Local re-implementation of the RES (Radiology Edit Score) leaderboard metric.

Written from the competition description in detail.md. The exact leaderboard
scorer is private, so a few details are best-effort guesses (flagged with
`# ASSUMPTION`). Use it for *relative* comparison of pipeline variants, and
keep absolute numbers in mind as optimistic/pessimistic.

RES_case = 0.65 * F + 0.35 * I
  F = field-aware FINDINGS score
  I = weighted word edit distance over the whole IMPRESSION section
"""

from __future__ import annotations

import re
import unicodedata

# --------------------------------------------------------------------------
# 1. Tokenisation / normalisation  (section: "Text normalization")
# --------------------------------------------------------------------------

# ASSUMPTION: the exact function-word and critical-word lists are not published.
FUNCTION_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "but", "by",
    "for", "from", "has", "have", "in", "into", "is", "it", "its", "of", "on",
    "or", "that", "the", "these", "this", "those", "to", "was", "were", "which",
    "with", "within", "there", "here", "notably", "again", "also",
}

CRITICAL_WORDS = {
    # negation
    "no", "not", "without", "absent", "negative", "unremarkable", "clear",
    # laterality
    "right", "left", "bilateral", "unilateral", "ipsilateral", "contralateral",
    # severity
    "mild", "moderate", "severe", "minimal", "minimally", "small", "large",
    "tiny", "trace", "marked", "extensive", "subtle", "significant", "slight",
    "slightly", "grossly", "borderline", "chronic", "acute",
    # laterality-ish / descriptors the spec calls clinically critical
    "focal", "diffuse", "stable", "new", "interval",
    # units
    "mm", "cm", "mm2", "cm2",
}

# ASSUMPTION: list of "common unit spellings" the description says are
# standardised. Extended a little for robustness.
UNIT_CANON = {
    "millimeters": "mm", "millimetres": "mm", "millimeter": "mm",
    "millimetre": "mm",
    "centimeters": "cm", "centimetres": "cm", "centimeter": "cm",
    "centimetre": "cm",
}

_LIST_MARKER = re.compile(r"^\s*(?:\d+\s*[.)]|[-*+•‣▪])\s+")
_SIGNED_NUM = re.compile(r"^[+-]\d")
_PUNCT = re.compile(r"[^\w\s+/-]|(?<=\w)/|/(?=\w)")


def normalize(text: str) -> str:
    """Lowercase, unicode-normalise, standardise units, strip list markers."""
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text).lower()
    for long, short in UNIT_CANON.items():
        text = re.sub(rf"\b{long}\b", short, text)
    out_lines = []
    for line in text.split("\n"):
        prev = None
        while prev != line:                      # markers can stack ("1. - ")
            prev = line
            line = _LIST_MARKER.sub("", line)
        out_lines.append(line)
    return "\n".join(out_lines)


def tokenize(text: str) -> list[str]:
    """Normalised tokens: punctuation stripped, list markers removed, hyphens
    between letters deleted ("air-space" -> "airspace") but hyphens touching a
    digit act as separators ("L5-S1" -> "l 5 s 1") so that glued and spaced
    spinal-level spellings compare equal. Letter/number boundaries are split.
    """
    text = normalize(text)
    # Keep + and - (the punctuation class above preserves them), stash signed
    # measurements next, then apply hyphen rules. The placeholder must be
    # letters-only so the letter/number boundary splits ignore it.
    text = _PUNCT.sub(" ", text)
    signed: list[str] = []

    def _stash(m):
        signed.append(m.group(0).replace(" ", ""))
        return " zzsig" + "a" * len(signed) + " "

    text = re.sub(r"[+\-]\s*\d+(?:\.\d+)?", _stash, text)
    text = re.sub(r"(?<=[a-z])-(?=[a-z])", "", text)   # air-space -> airspace
    text = re.sub(r"[+\-]", " ", text)                  # L5-S1 -> L 5 S 1
    text = re.sub(r"(?<=[a-z])(?=\d)", " ", text)       # t9 -> t 9
    text = re.sub(r"(?<=\d)(?=[a-z])", " ", text)       # 7th -> 7 th
    out = []
    for t in text.split():
        m = re.fullmatch(r"zzsig(a*)", t)
        out.append(signed[len(m.group(1)) - 1] if m else t)  # "+3" as one token
    return [t for t in out if t]


# --------------------------------------------------------------------------
# 2. Weighted word edit distance
# --------------------------------------------------------------------------

W_CRITICAL, W_CONTENT, W_FUNCTION = 4.0, 2.0, 0.25


def token_weight(tok: str) -> float:
    if tok in CRITICAL_WORDS or tok in UNIT_CANON.values():
        return W_CRITICAL
    if tok in FUNCTION_WORDS:
        return W_FUNCTION
    if any(ch.isdigit() for ch in tok):
        return W_CRITICAL                       # measurements
    return W_CONTENT


def _total_weight(tokens: list[str]) -> float:
    return sum(token_weight(t) for t in tokens)


def weighted_word_edit(ref: str, sub: str) -> float:
    """Ordered weighted Levenshtein, normalised by the larger token weight."""
    a, b = tokenize(ref), tokenize(sub)
    wa, wb = _total_weight(a), _total_weight(b)
    denom = max(wa, wb)
    if denom == 0:
        return 0.0
    n, m = len(a), len(b)
    # cost[i][j] = min cost to transform a[:i] into b[:j]
    prev = [0.0] * (m + 1)
    for j in range(1, m + 1):
        prev[j] = prev[j - 1] + token_weight(b[j - 1])          # insert
    for i in range(1, n + 1):
        cur = [prev[0] + token_weight(a[i - 1])]                 # delete
        wa_i = token_weight(a[i - 1])
        for j in range(1, m + 1):
            sub_cost = 0.0 if a[i - 1] == b[j - 1] else max(
                wa_i, token_weight(b[j - 1])
            )
            cur.append(min(prev[j] + wa_i,      # delete
                           cur[j - 1] + token_weight(b[j - 1]),   # insert
                           prev[j - 1] + sub_cost))               # substitute
        prev = cur
    return min(1.0, prev[m] / denom)


# --------------------------------------------------------------------------
# 3. Section / field parsing
# --------------------------------------------------------------------------

# Labels are matched case-insensitively: real templates contain BOTH "BONES:"
# and "Bones:", and the scorer lowercases everything before comparing, so the
# two must align. Getting this wrong silently drops whole fields.
_LABEL = re.compile(r"^([A-Za-z][A-Za-z0-9 /\-&.,()']{1,70}):\s*(.*)$")
RESERVED_LABELS = {"FINDINGS", "IMPRESSION"}
_LABEL_STRIP = re.compile(r"[^A-Z0-9]+")


def canonical(label: str) -> str:
    """Case/space/punctuation-insensitive key for aligning fields by label."""
    return _LABEL_STRIP.sub("", (label or "").upper())


def split_sections(text: str) -> tuple[str, str, str]:
    """-> (findings_block, impression_block, trailing_block)"""
    text = text if isinstance(text, str) else ""
    m = re.search(r"(?m)^\s*FINDINGS\s*:", text)
    if not m:
        return "", text, ""
    head = text[: m.start()]
    rest = text[m.end():]
    mi = re.search(r"(?m)^\s*IMPRESSION\s*:", rest)
    if not mi:
        return rest, "", ""
    findings = rest[: mi.start()]
    impression = rest[mi.end():]
    # Anything after IMPRESSION that is not part of it (e.g. LUNG-RADS block)
    # is treated as trailing and is ignored by the scorer.
    return findings, impression, ""


def parse_fields(block: str) -> tuple[dict[str, str], dict[str, str], str]:
    """Split a FINDINGS block.

    Returns (fields, display, unlabelled) where
      fields    : {canonical_label: content}
      display   : {canonical_label: label exactly as written}
      unlabelled: free text sitting directly under FINDINGS:
    """
    fields: dict[str, str] = {}
    display: dict[str, str] = {}
    unlabelled: list[str] = []
    current: str | None = None
    buf: list[str] = []

    def flush():
        if current is not None:
            fields[current] = " ".join(x.strip() for x in buf).strip()

    for raw in (block or "").split("\n"):
        line = raw.strip()
        if not line:
            continue
        m = _LABEL.match(line)
        if m and m.group(1).strip().upper() not in RESERVED_LABELS:
            flush()
            current = canonical(m.group(1))
            display.setdefault(current, m.group(1).strip())
            buf = [m.group(2)]
        elif current is None:
            unlabelled.append(line)
        else:
            buf.append(line)
    flush()
    return fields, display, " ".join(unlabelled).strip()


# --------------------------------------------------------------------------
# 4. RES
# --------------------------------------------------------------------------

def findings_score(ref_f: dict[str, str], sub_f: dict[str, str],
                   tpl_f: dict[str, str], ref_disp: dict | None = None
                   ) -> tuple[float, dict]:
    """
    F = sum(field_weight * field_word_edit) / sum(field_weight)

    field_weight = 3 if the reference field differs from the template field
                    1 if the reference field is unchanged from the template
    Missing expected field -> compare reference content against "".
    Unexpected fields / unlabelled content -> extra penalty (ASSUMPTION: added
    as a weight-1 pseudo-field, which mirrors "additional penalties").
    """
    ref_disp = ref_disp or {}
    num = den = 0.0
    detail = {}
    for label, ref_txt in ref_f.items():
        tpl_txt = tpl_f.get(label, "")
        changed = normalize(ref_txt).strip() != normalize(tpl_txt).strip()
        w = 3.0 if changed else 1.0
        sub_txt = sub_f.get(label)
        e = 1.0 if sub_txt is None else weighted_word_edit(ref_txt, sub_txt)
        num += w * e
        den += w
        detail[label] = {"w": w, "edit": round(e, 3),
                         "missing": sub_txt is None,
                         "ref": ref_txt[:90],
                         "ref_display": ref_disp.get(label, label)}

    # unexpected fields: penalised as extra content
    for label, sub_txt in sub_f.items():
        if label in ref_f:
            continue
        e = weighted_word_edit(sub_txt, "") if sub_txt.strip() else 0.0
        num += 1.0 * e
        den += 1.0
        detail[label] = {"w": 1.0, "edit": round(e, 3), "unexpected": True,
                         "sub": sub_txt[:90]}

    if den == 0:
        return 0.0, detail
    return min(1.0, num / den), detail


def res_case(ref_report: str, sub_report: str, template: str) -> dict:
    r_fb, r_im, _ = split_sections(ref_report)
    s_fb, s_im, _ = split_sections(sub_report)
    t_fb, _t_im, _t_tr = split_sections(template)
    ref_f, ref_disp, _ = parse_fields(r_fb)
    sub_f, _sd, sub_unlab = parse_fields(s_fb)
    tpl_f, _td, _ = parse_fields(t_fb)

    F, detail = findings_score(ref_f, sub_f, tpl_f, ref_disp)
    I = weighted_word_edit(r_im, s_im)
    return {"F": F, "I": I, "RES": 0.65 * F + 0.35 * I, "fields": detail,
            "unlabelled": sub_unlab}


def res_dataset(refs, subs, templates) -> dict:
    """refs/subs/templates: pandas Series aligned by index."""
    rows = [res_case(r, s, t) for r, s, t in zip(refs, subs, templates)]
    F = sum(x["F"] for x in rows) / len(rows)
    I = sum(x["I"] for x in rows) / len(rows)
    return {"RES": 0.65 * F + 0.35 * I, "F": F, "I": I, "rows": rows}


if __name__ == "__main__":
    ref = """FINDINGS:
LUNGS: Mild right basilar airspace opacity. No pulmonary edema.
PLEURA: Small right pleural effusion. No pneumothorax.
OSSEOUS STRUCTURES: No acute osseous abnormality identified on these views.

IMPRESSION:
Mild right basilar airspace opacity and small right pleural effusion."""
    tpl = """FINDINGS:
LUNGS: No focal airspace opacity or pulmonary edema.
PLEURA: No pleural effusion or pneumothorax.
OSSEOUS STRUCTURES: No acute osseous abnormality identified on these views.

IMPRESSION:
No acute cardiopulmonary abnormality."""
    print("perfect ->", res_case(ref, ref, tpl)["RES"])
    print("template as-is ->", round(res_case(ref, tpl, tpl)["RES"], 4))
    print("dictation only ->", round(res_case(ref, "mild right basilar "
          "opacity, small right pleural effusion", tpl)["RES"], 4))

In [ ]:
%%writefile src/natoe/pipeline.py
"""
Radiology dictation -> structured report pipeline.

Design (see notes in the accompanying notebook):

  1. RETRIEVE  2-3 similar train cases (same body_part / modality family) so the
     model sees the house style used for that exact template shape.
  2. EXTRACT   the LLM returns a JSON *patch*: only the FINDINGS fields that
     change, plus the IMPRESSION text. It never emits the whole report.
  3. GUARD     deterministic post-checks strip unsupported numbers /
     laterality / severity and any hallucinated field label.
  4. RENDER    Python assembles the report from the template, so field labels,
     field order and untouched normal statements are preserved byte-for-byte.

The LLM does the language work; the code does the bookkeeping. That split is
what keeps RES low, because RES is a template-edit metric.
"""

from __future__ import annotations

import hashlib
import json
import math
import os
import re
import threading
import time
from collections import Counter

import pandas as pd

from .config import PROVIDERS
from .res_scorer import (split_sections, parse_fields, normalize, tokenize,
                         canonical, _LIST_MARKER)

__all__ = [
    "PROVIDERS", "Retriever", "SYSTEM_PROMPT", "LLM", "MockLLM", "Cache",
    "clip", "build_user_prompt", "parse_json", "_sentences", "unsupported",
    "guard", "render", "gold_patch", "gold_json", "build_shots", "run_case",
    "run_dataset",
]

# --------------------------------------------------------------------------
# 1. Few-shot retrieval  (dependency-free TF-IDF)
# --------------------------------------------------------------------------

MAX_CHARS_TEMPLATE = 4000      # truncate the huge Lung-RADS boilerplate
MAX_CHARS_DICTATION = 6000

def _norm_key(s: str) -> str:
    return " ".join(tokenize(s))


class Retriever:
    """TF-IDF cosine over normalised tokens, cosine similarity.

    Dependency-free on purpose: the corpus is only a few hundred rows, so
    numpy-free sparse dot products are more than fast enough and keep the
    notebook self-contained.
    """

    def __init__(self, corpus: list[str]):
        self.docs = [_norm_key(c) for c in corpus]
        self.n = max(1, len(self.docs))
        df: Counter = Counter()
        for d in self.docs:
            df.update(set(d.split()))
        # smoothed inverse document frequency
        self.idf = {t: math.log(self.n / (1 + c)) + 1.0 for t, c in df.items()}
        self.max_idf = math.log(self.n) + 1.0
        self.vecs = [self._vec(d) for d in self.docs]

    def _vec(self, doc: str) -> dict[str, float]:
        v = {t: (1.0 + math.log(c)) * self.idf.get(t, self.max_idf)
             for t, c in Counter(doc.split()).items()}
        norm = math.sqrt(sum(x * x for x in v.values())) or 1.0
        return {t: x / norm for t, x in v.items()}

    def query(self, text: str, top_k: int = 3, exclude: set | None = None) -> list[int]:
        q = self._vec(_norm_key(text))
        scored = []
        for i, v in enumerate(self.vecs):
            if exclude and i in exclude:
                continue
            if len(q) > len(v):                      # iterate the smaller side
                s = sum(q[t] * v.get(t, 0.0) for t in q)
            else:
                s = sum(v[t] * q.get(t, 0.0) for t in v)
            scored.append((s, i))
        scored.sort(key=lambda p: (-p[0], p[1]))
        return [i for _, i in scored[:top_k]]


# --------------------------------------------------------------------------
# 2. Prompt
# --------------------------------------------------------------------------

SYSTEM_PROMPT = """\
You are a radiology report editor. You convert a radiologist's dictation into an \
edit of a NORMAL reference template. The metric scores how faithfully you edit \
the template, so copying exact wording matters far more than being elegant.

Return ONE JSON object and nothing else:

{
  "fields": {
    "<EXACT TEMPLATE LABEL>": "<new content for that field>"
  },
  "impression": "<the full IMPRESSION section text>"
}

HARD RULES
1. Copy the radiologist's sentences VERBATIM into the matching field. Never \
paraphrase, never tidy grammar, never "improve" wording. Numbers, laterality \
and severity must match the dictation exactly.
2. Two normalisations, and only these two. Everything else is copied as-is:
   a) EXPAND spinal-level shorthand to the canonical hyphenated form: "c56" \
and "c6 7" and "c5-6, c7" all become "C5-C6", "C6-C7", "C5-C6 and C6-C7". \
Same for L3-4 -> L3-L4, T12-13 -> T12-T13. The reference reports do this in \
87% of cases.
   b) KEEP ordinary misspellings exactly as dictated. The reference reports \
reproduce the dictation's spelling errors verbatim in 100% of cases, so \
"ostearth", "andd", "dspaces" must appear in your output unchanged. Fixing a \
spelling error is scored as an edit. Do not expand abbreviations, do not \
reformat numbers, do not tidy grammar.
3. Only include a field in "fields" if you are CHANGING it. Fields the \
dictation says nothing about must be left out entirely — the renderer keeps \
the template's original text for them.
4. Use ONLY the field labels that appear in the template, spelled exactly as \
they appear (including slashes and spaces). Never invent a new field.
5. Route each finding to the field that owns that anatomy, and route it to \
exactly ONE field. An opacity goes in a lung field, an effusion in a pleural \
field, degenerative change in a bone or disc field, a fracture in a bone \
field. If the template has an "OTHER FINDINGS" field, use it only for \
findings that belong nowhere else. Getting this wrong is the single most \
expensive mistake you can make.
6. When a field is abnormal, DELETE the template's contradicting normal \
sentence (e.g. do not keep "The lungs are clear" next to an opacity) but KEEP \
any normal sentence that stays true and was not contradicted. Copy those kept \
normals out of the template word for word.
7. If the dictation explicitly negates something the template states as normal, \
write the negated form and keep the surrounding normal wording.
8. IMPRESSION rules, in priority order:
   a) If the dictation describes NO abnormality (it is normal, or it only \
contains technique/positioning notes such as "no acute traumatic process", \
"external marker projects over", "exposure adequate"), leave the template's \
IMPRESSION completely unchanged. Never put a technical note in the \
IMPRESSION.
   b) Otherwise the IMPRESSION is a short numbered or plain list of the \
important abnormal findings, built from the dictation's own summary wording.
9. Never add a finding, measurement, diagnosis or history that is not in the \
dictation. Do not comment on the images yourself.

EXAMPLE OF THE REQUIRED EDIT STYLE
Dictation: "no effusuon, infiltrates / mild thoracic spondylosis"
Template BONES field: "No acute osseous abnormality identified on this examination."
Edited BONES field: "Mild thoracic spondylosis is present. No acute osseous \
abnormality identified on this examination."
The new abnormality is prepended; the still-true normal sentence is kept.
Note the misspelling "effusuon" is left alone, per rule 2b.
"""


def build_user_prompt(case: dict, shots: list[dict]) -> str:
    parts = []
    for s in shots:
        parts.append(
            "### EXAMPLE\n"
            f"TEMPLATE:\n{clip(s['template_content'])}\n\n"
            f"DICTATION:\n{clip(s['dictation'])}\n\n"
            f"JSON OUTPUT:\n{s['_gold_json']}\n"
        )

    tpl = clip(case["template_content"])
    _f, display, _u = parse_fields(tpl)
    labels = [display[k] for k in _f]
    parts.append(
        "### NOW EDIT THIS CASE\n"
        f"modality: {case['modality']}\n"
        f"body_part: {case['body_part']}\n"
        f"study: {case['study_description']}\n"
        f"age_band: {case['patient_age_band']}\n"
        f"sex: {case['patient_sex']}\n\n"
        f"TEMPLATE FIELD LABELS (use these exactly): {json.dumps(labels)}\n\n"
        f"TEMPLATE:\n{tpl}\n\n"
        f"DICTATION:\n{clip(case['dictation'])}\n\n"
        "JSON OUTPUT:"
    )
    return "\n\n".join(parts)


def clip(text: str, n: int = MAX_CHARS_TEMPLATE) -> str:
    text = "" if text is None else str(text)
    if len(text) <= n:
        return text
    # keep the head (FINDINGS + IMPRESSION) and drop long trailing boilerplate
    return text[:n] + "\n...[truncated]"


# --------------------------------------------------------------------------
# 3. LLM client (OpenAI-compatible: Groq / OpenRouter / anything)
# --------------------------------------------------------------------------

# Reasoning models burn most of their output budget on chain-of-thought. For
# this task the reasoning is not the bottleneck, so keep it minimal.
REASONING_MODELS = ("gpt-oss", "nemotron", "qwen3", "space-bunny", "big-pickle",
                    "muse-spark", "mimo", "longcat")


def _is_fatal(exc: Exception) -> bool:
    """True when retrying the same request cannot help (bad model id, bad param)."""
    status = getattr(exc, "status_code", None)
    if status in (400, 401, 403, 404):
        return True
    msg = str(exc).lower()
    return any(k in msg for k in ("model_not_found", "does not exist",
                                  "invalid_api_key", "not supported",
                                  "unsupported_parameter",
                                  "reasoning_effort is not supported"))


def _is_rate_limited(exc: Exception) -> bool:
    return getattr(exc, "status_code", None) == 429 or \
        "429" in str(exc) or "rate limit" in str(exc).lower()


def _backoff(exc: Exception, attempt: int) -> float:
    """Wait before retrying.

    Rate limits get a much longer wait, because the Groq token budget
    (8000 prompt tokens/min) needs a whole window to refill, and retrying
    early just burns the remaining quota.
    """
    if not _is_rate_limited(exc):
        return 2.0 * (2 ** attempt)
    delay = 20.0 * (2 ** attempt)
    resp = getattr(exc, "response", None)
    if resp is not None:
        hint = None
        try:
            hint = resp.headers.get("retry-after")
        except Exception:                                 # noqa: BLE001
            hint = None
        if hint:
            try:
                delay = max(delay, float(hint))
            except (TypeError, ValueError):
                pass
    return min(delay, 120.0)


class LLM:
    def __init__(self, provider: str | None = None, model: str | None = None,
                 reasoning_effort: str = "low"):
        self.provider = (provider or os.getenv("LLM_PROVIDER", "groq")).lower()
        cfg = PROVIDERS.get(self.provider)
        if cfg is None:
            raise ValueError(f"unknown provider {self.provider!r}; "
                             f"choose from {list(PROVIDERS)}")
        self.cfg = cfg
        key = os.getenv(cfg["key_env"], "").strip()
        if not key:
            raise RuntimeError(f"{cfg['key_env']} is not set. Put it in .env")
        self.api_key = key
        env_model = os.getenv("LLM_MODEL", "").strip()
        self.model = model or env_model or cfg["default_model"]
        self.chain = [self.model] + [m for m in cfg["fallbacks"]
                                     if m != self.model]
        self.reasoning_effort = (os.getenv("REASONING_EFFORT", reasoning_effort)
                                 or "").strip()
        # A hard socket timeout matters more than a generous one: a hung
        # request otherwise stalls a whole worker for minutes and the row
        # ends up on the template fallback.
        self.timeout = float(os.getenv("LLM_TIMEOUT", "90"))
        from openai import OpenAI
        self.client = OpenAI(base_url=cfg["base_url"], api_key=self.api_key,
                             timeout=self.timeout, max_retries=0)

    def _is_reasoning(self, model: str) -> bool:
        return any(k in model.lower() for k in REASONING_MODELS)

    def complete(self, prompt: str, system: str = SYSTEM_PROMPT,
                 max_tokens: int = 2000, temperature: float = 0.0,
                 attempts: int = 3) -> str:
        """Return raw model text.

        Retries the same model with exponential backoff (free tiers rate-limit
        aggressively under concurrency), then walks the fallback model list.
        Degrades gracefully along the way: drops JSON mode if the endpoint
        rejects it, and if a reasoning model spends its whole budget on
        chain-of-thought and returns no `content`, salvages the JSON object out
        of the `reasoning` field instead.
        """
        last: Exception | None = None
        errors: list[str] = []
        for model in self.chain:
            for json_mode in (True, False):
                for attempt in range(attempts):
                    try:
                        kw: dict = dict(
                            model=model,
                            messages=[{"role": "system", "content": system},
                                      {"role": "user", "content": prompt}],
                            temperature=temperature,
                            max_tokens=max_tokens,
                        )
                        if json_mode:
                            kw["response_format"] = {"type": "json_object"}
                        if self._is_reasoning(model) and self.reasoning_effort:
                            kw["reasoning_effort"] = self.reasoning_effort
                        msg = self.client.chat.completions.create(
                            **kw).choices[0].message
                        content = (getattr(msg, "content", None) or "").strip()
                        if content and parse_json_safe(content) is not None:
                            return content
                        # Reasoning models put the chain of thought in one of
                        # two fields depending on the gateway. If the content
                        # is empty or truncated mid-object, salvage the JSON
                        # out of the reasoning trace instead of giving up.
                        for attr in ("reasoning_content", "reasoning"):
                            blob = (getattr(msg, attr, None) or "").strip()
                            if not blob:
                                continue
                            salvaged = _last_json_object(blob)
                            if salvaged and parse_json_safe(salvaged) is not None:
                                return salvaged
                        if content:
                            return content          # let the caller report it
                        last = RuntimeError("empty content and reasoning")
                        break                      # nothing to retry
                    except Exception as e:       # noqa: BLE001
                        last = e
                        if _is_fatal(e):
                            break              # this model/flag combo is wrong
                        time.sleep(_backoff(e, attempt))
                errors.append(f"{model}[json={json_mode}]: "
                              f"{type(last).__name__}: {str(last)[:120]}")
        raise RuntimeError("all models failed -> " + " | ".join(errors[:4]))


def _last_json_object(text: str) -> str | None:
    """Pull the last balanced {...} out of a blob, ignoring braces in strings."""
    depth = start = 0
    best = None
    in_str = escape = False
    for i, ch in enumerate(text):
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = in_str
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                best = text[start:i + 1]
    return best


def parse_json_safe(txt: str):
    """parse_json that returns None instead of raising."""
    try:
        return parse_json(txt)
    except Exception:                                       # noqa: BLE001
        return None


class MockLLM:
    """Offline stand-in so the notebook can be smoke-tested without a key.

    Crude but real: it routes each dictation sentence to the template field
    whose anatomy keywords it matches best. The score is meaningless -- it only
    proves the plumbing (prompt -> JSON -> guard -> render) runs end to end.
    """

    provider, cfg, api_key = "mock", {}, ""
    model, chain = "mock", ["mock"]
    reasoning_effort = ""      # MockLLM must quack like LLM for the cache key

    @staticmethod
    def _keys(label: str) -> set[str]:
        stop = {"no", "the", "and", "of", "is", "are", "normal", "unremarkable",
                "finding", "findings", "structures", "space", "spaces"}
        return {w for w in re.findall(r"[a-z]{4,}", label.lower())
                if w not in stop}

    def complete(self, prompt, system=None, max_tokens=2000, temperature=0.0):
        tpl = re.search(r"TEMPLATE:\n(.*?)\n\nDICTATION:", prompt, re.S)
        dct = re.search(r"DICTATION:\n(.*?)\n\nJSON OUTPUT:", prompt, re.S)
        labs = re.search(r"FIELD LABELS \(use these exactly\): (\[.*?\])",
                         prompt, re.S)
        if not (dct and labs):
            return '{"fields": {}, "impression": ""}'
        sents = _sentences(dct.group(1))
        fields = {}
        for lab in json.loads(labs.group(1)):
            keys = self._keys(lab)
            if not keys:
                continue
            best, hits_best = "", 0
            for s in sents:
                hits = len(keys & set(re.findall(r"[a-z]{4,}", s.lower())))
                if hits > hits_best:
                    best, hits_best = s, hits
            if hits_best >= 1:
                fields[lab] = best
        return json.dumps({"fields": fields,
                           "impression": "\n".join(sents[:3])},
                          ensure_ascii=False)


# --------------------------------------------------------------------------
# 4. Parse + guard the model patch
# --------------------------------------------------------------------------

_FENCE = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$", re.M)

LATERALITY = {"right", "left", "bilateral", "unilateral", "ipsilateral",
              "contralateral"}
# Comparative qualifiers only. Deliberately excludes "acute", "chronic" and
# "significant": they appear in almost every normal template sentence, so
# flagging them would delete faithful text.
SEVERITY = {"mild", "moderate", "severe", "minimal", "minimally", "small",
            "large", "tiny", "trace", "marked", "extensive", "subtle",
            "slight", "slightly", "grossly", "borderline"}


def parse_json(txt: str) -> dict:
    txt = _FENCE.sub("", txt or "").strip()
    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        raise ValueError("no JSON object in response")
    return json.loads(m.group(0))


def _sentences(text: str) -> list[str]:
    """Split into sentences line by line, keeping list items ("1. foo") glued
    to their first sentence so numbered IMPRESSION lines survive a round trip.
    """
    out: list[str] = []
    for line in (text or "").split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue
        head = ""
        m = _LIST_MARKER.match(line)
        if m:
            head, line = m.group(0), line[m.end():]
        parts = [p.strip() for p in
                 re.split(r"(?<=[.;:])\s+(?=[A-Z0-9(])", line) if p.strip()]
        for i, p in enumerate(parts):
            out.append(head + p if i == 0 else p)
    return out


def unsupported(sent: str, allowed: set[str], level: str = "numbers") -> list[str]:
    """Critical tokens in `sent` that have no support in `allowed`.

    `allowed` is the dictation's token set UNION the field's original template
    text, so faithfully preserved normal sentences survive.

    level:
      "off"     - no check
      "numbers" - only new measurements are flagged. A measurement is a
                  concrete factual claim, so inventing one is the clearest
                  form of unsupported content. This is the default.
      "strict"  - also flag new laterality / severity words.
    Negation is never used as a trigger: the template legitimately says
    "No pneumothorax." even when the dictation never mentioned pneumothorax.
    """
    if level == "off":
        return []
    out = []
    for t in tokenize(sent):
        if t in allowed:
            continue
        if any(c.isdigit() for c in t):
            out.append(t)                                   # measurements
        elif level == "strict" and (t in LATERALITY or t in SEVERITY):
            out.append(t)
    return out


def guard(patch: dict, case: dict, level: str = "numbers") -> dict:
    """Drop hallucinated field labels and unsupported sentences."""
    tpl_fields, _tpl_display, _ = parse_fields(
        split_sections(case["template_content"])[0])
    dict_tok = set(tokenize(case["dictation"]))
    tpl_imp_tok = set(tokenize(split_sections(case["template_content"])[1]))
    drops = []

    clean = {}
    for label, content in (patch.get("fields") or {}).items():
        key = canonical(label)
        if key not in tpl_fields:               # tolerate slug/case slips
            cands = [k for k in tpl_fields if key and (key in k or k in key)]
            if cands:
                key = min(cands, key=len)
            else:
                drops.append(f"label:{label}")
                continue
        allowed = dict_tok | set(tokenize(tpl_fields[key]))
        sents = _sentences(content)
        keep = [s for s in sents if not unsupported(s, allowed, level)]
        for s in sents:
            if s not in keep:
                drops.append(f"unsupported:{s[:40]}")
        if keep:
            clean[key] = " ".join(keep)

    imp = patch.get("impression") or ""
    imp_allowed = dict_tok | tpl_imp_tok
    imp_sents = _sentences(imp)
    imp_keep = [s for s in imp_sents if not unsupported(s, imp_allowed, level)]
    imp_clean = "\n".join(imp_keep)
    if imp.strip() and not imp_keep:
        drops.append("impression_fully_unsupported")

    return {"fields": clean, "impression": imp_clean, "drops": drops}


# --------------------------------------------------------------------------
# 5. Render the final report (pure Python, template-faithful)
# --------------------------------------------------------------------------

def render(case: dict, clean: dict) -> str:
    tpl_fields, tpl_display, _ = parse_fields(
        split_sections(case["template_content"])[0])
    tpl_imp = split_sections(case["template_content"])[1]

    lines = []
    for key in tpl_fields:                      # template order, always
        label = tpl_display[key]
        content = clean["fields"].get(key, tpl_fields[key]).strip()
        lines.append(f"{label}: {content}".rstrip())

    impression = (clean["impression"] or "").strip() or tpl_imp.strip()
    return "FINDINGS:\n" + "\n".join(lines) + "\n\nIMPRESSION:\n" + impression


# --------------------------------------------------------------------------
# 6. Gold patch builder (used to create few-shot exemplars from train.csv)
# --------------------------------------------------------------------------

def gold_patch(row) -> dict:
    ref_fields, _d, _u = parse_fields(split_sections(row.report)[0])
    tpl_fields, _td, _tu = parse_fields(split_sections(row.template_content)[0])
    changed = {}
    for k, v in ref_fields.items():
        if k in tpl_fields and normalize(v).strip() != \
                normalize(tpl_fields[k]).strip():
            changed[k] = v
    return {
        "fields": changed,
        "impression": split_sections(row.report)[1].strip(),
    }


def gold_json(row) -> str:
    p = gold_patch(row)
    # keep exemplars short: drop fields whose gold edit is huge
    p["fields"] = {k: v for k, v in list(p["fields"].items())[:6]}
    return json.dumps(p, ensure_ascii=False, indent=1)


# --------------------------------------------------------------------------
# 7. Runner (with an on-disk cache so prompt iteration is cheap)
# --------------------------------------------------------------------------

class Cache:
    """On-disk response cache, safe for concurrent writers.

    Writes are serialised by a lock and land via an atomic replace, so a run
    that is killed mid-write can never leave a truncated cache behind.
    """

    def __init__(self, path: str | None):
        self.path = path
        self._lock = threading.Lock()
        self.data: dict[str, str] = {}
        if path and os.path.exists(path):
            try:
                with open(path) as fh:
                    self.data = json.load(fh)
            except (json.JSONDecodeError, OSError):
                self.data = {}

    @staticmethod
    def key(*parts) -> str:
        return hashlib.sha1("\x00".join(map(str, parts)).encode()).hexdigest()

    def get(self, k):
        with self._lock:
            return self.data.get(k)

    def put(self, k, v):
        # Never cache a failure. A transient 429 that got cached would poison
        # every later run of the same config.
        if not v:
            return
        with self._lock:
            self.data[k] = v
            if not self.path:
                return
            tmp = f"{self.path}.tmp"
            try:
                with open(tmp, "w") as fh:
                    json.dump(self.data, fh)
                os.replace(tmp, self.path)          # atomic
            except OSError:
                pass


def build_shots(retriever: Retriever, train: pd.DataFrame, case: dict,
                n_shots: int = 3) -> list[dict]:
    """Nearest training cases, as ready-to-print exemplars."""
    if n_shots <= 0:
        return []
    query = f"{case['body_part']} {case['modality']} {case['study_description']} " \
            f"{case['template_content']} {case['dictation']}"
    idx = retriever.query(query, top_k=n_shots)
    shots = []
    for i in idx:
        row = train.iloc[i]
        shots.append({
            "template_content": row.template_content,
            "dictation": row.dictation,
            "_gold_json": gold_json(row),
        })
    return shots


def run_case(llm, case: dict, shots: list[dict],
             guard_level: str = "numbers") -> tuple[str, dict]:
    prompt = build_user_prompt(case, shots)
    raw = llm.complete(prompt)
    try:
        patch = parse_json(raw)
    except (ValueError, json.JSONDecodeError):
        patch = {}                      # unparseable -> fall back to template
    clean = guard(patch, case, guard_level)
    return render(case, clean), clean


def run_dataset(llm, df: pd.DataFrame, retriever: Retriever,
                train: pd.DataFrame, n_shots: int = 3,
                guard_level: str = "numbers", cache_path: str | None = None,
                progress: bool = True, workers: int = 2,
                retries: int = 2) -> list[str]:
    """Predict every row. Responses are cached on disk keyed by
    (model, reasoning_effort, guard_level, n_shots, system_prompt, prompt) so
    prompt iteration is nearly free.

    `workers` and `retries` exist because of rate limiting, not throughput.
    Free tiers ration *prompt tokens* (Groq: 8000/min), so a burst of parallel
    few-shot requests gets 429s and silently degrades the score to
    "template unedited". Failing loudly and retrying slowly is worth far more
    than raw speed here.
    """
    from concurrent.futures import ThreadPoolExecutor

    cache = Cache(cache_path)
    cases, prompts = [], []
    for _, r in df.iterrows():
        case = {"template_content": r.template_content, "dictation": r.dictation,
                "modality": r.modality, "body_part": r.body_part,
                "study_description": r.study_description,
                "patient_age_band": r.patient_age_band,
                "patient_sex": r.patient_sex}
        shots = build_shots(retriever, train, case, n_shots)
        prompt = build_user_prompt(case, shots)
        cases.append(case)
        prompts.append(prompt)

    failures: list[str] = []

    def fetch(args):
        i, prompt = args
        # The system prompt MUST be part of the key: otherwise editing the
        # prompt rules would silently reuse every stale cached response.
        ck = Cache.key(llm.model, llm.reasoning_effort, guard_level, n_shots,
                       SYSTEM_PROMPT, prompt)
        raw = cache.get(ck)
        if raw is None:
            try:
                raw = llm.complete(prompt)
            except Exception as e:                       # noqa: BLE001
                failures.append(f"row {i}: {type(e).__name__}: {str(e)[:160]}")
                return ""
            cache.put(ck, raw)
        return raw

    t0 = time.time()
    n = len(cases)
    indices = list(range(1, n + 1))
    raws: list[str | None] = [None] * n

    def sweep(idxs: list[int], label: str) -> None:
        if not idxs:
            return
        if progress:
            print(f"  {label}: {len(idxs)} row(s)")
        with ThreadPoolExecutor(max_workers=workers) as pool:
            # pool.map preserves input order, so zipping back onto idxs is safe.
            results = pool.map(fetch, [(i, prompts[i - 1]) for i in idxs])
            for done, (i, raw) in enumerate(zip(idxs, results), 1):
                raws[i - 1] = raw
                if progress and (done % 10 == 0 or done == len(idxs)):
                    _tick(done, len(idxs), t0)

    sweep(indices, "pass 1")
    for attempt in range(1, retries + 1):
        missing = [i for i in indices if not raws[i - 1]]
        if not missing:
            break
        # Failures are never cached, so a retry costs a fresh call and a
        # second chance is always worth taking.
        time.sleep(30.0 * attempt)
        failures.clear()
        sweep(missing, f"retry {attempt}")

    if progress:
        print()
    if failures:
        print(f"  WARNING: {len(failures)} row(s) exhausted retries and fell "
              f"back to the template. First 5:")
        for f in failures[:5]:
            print(f"    {f}")

    outs = []
    for case, raw in zip(cases, raws):
        try:
            patch = parse_json(raw) if raw else {}
        except (ValueError, json.JSONDecodeError):
            patch = {}                   # unparseable -> fall back to template
        outs.append(render(case, guard(patch, case, guard_level)))
    return outs


def _tick(done: int, total: int, t0: float) -> None:
    rate = done / max(1e-9, time.time() - t0)
    print(f"  {done}/{total}  {rate:.2f} rows/s  "
          f"eta {(total - done) / max(1e-9, rate):.0f}s")

In [ ]:
%%writefile src/natoe/config.py
"""Configuration and shared paths.

Model choices are read from the environment so that nothing sensitive ever
lives in code or in a notebook.

Primary:  Groq, `openai/gpt-oss-120b`      (LLM_PROVIDER=groq)
Fallback: OpenRouter, a `:free` model id   (LLM_PROVIDER=openrouter)
"""

from __future__ import annotations

import os
from pathlib import Path

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:                                    # pragma: no cover
    pass

# --------------------------------------------------------------------- paths

ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = ROOT / "data"
NOTEBOOK_DIR = ROOT / "notebooks"
OUTPUT_DIR = ROOT / "outputs"
CACHE_PATH = ROOT / "outputs" / "llm_cache.json"

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"

INPUT_COLUMNS = [
    "case_id", "modality", "body_part", "study_description",
    "patient_age_band", "patient_sex", "template_content", "dictation",
]

# ------------------------------------------------------------------ providers

# Both providers expose an OpenAI-compatible /chat/completions API, so one
# client covers both. `fallbacks` is walked in order so a single retired model
# id cannot kill a run.
PROVIDERS: dict[str, dict] = {
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "key_env": "GROQ_API_KEY",
        "default_model": "openai/gpt-oss-120b",
        # Verified against GET /models on Groq (2026-09). The free tier is
        # aggressively rate-limited, so keep the chain short and the
        # concurrency low.
        "fallbacks": [
            "openai/gpt-oss-120b",
            "openai/gpt-oss-20b",
            "qwen/qwen3.8-27b",
        ],
    },
    "opencode": {
        "base_url": "https://opencode.ai/zen/v1",
        "key_env": "OPENCODE_API_KEY",
        # NOTE: Zen model ids carry NO "opencode/" prefix on the wire.
        #
        # Verified 2026-09-27 against this account:
        #   * Paid ids (gpt-5.5, claude-sonnet-5, gemini-3.8-flash, ...) return
        #     402 "Insufficient account balance" -- the key is valid but the
        #     account has no credits.
        #   * The `-free` ids return 403 "OpenCode's free tier can only be used
        #     from within OpenCode", so they are not callable from a script.
        #   * `space-bunny-free` is the exception and does work from a script.
        # Set LLM_MODEL to any paid id once the account has credits.
        "default_model": "space-bunny-free",
        "fallbacks": [
            "space-bunny-free",
            "muse-spark-1.3-contributor-free",
            "nemotron-3-ultra-free",
            "big-pickle",
        ],
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "key_env": "OPENROUTER_API_KEY",
        # Verified on the OpenRouter :free catalogue. nemotron-3-super is the
        # default because it is the only free model that honours JSON mode and
        # answers in well under a second; gemma-4-31b:free frequently 429s.
        "default_model": "nvidia/nemotron-3-super-120b-a12b:free",
        "fallbacks": [
            "nvidia/nemotron-3-super-120b-a12b:free",
            "google/gemma-4-31b-it:free",
            "qwen/qwen3.8-27b:free",
        ],
    },
}

DEFAULT_GUARD_LEVEL = "numbers"
# 0 shots, not 3. Both free tiers ration *prompt tokens*, and at 1 shot 6 of 40
# dev rows were rate-limited into a template fallback, which is worse than not
# prompting at all. Measured: RES 0.4366 at 0 shots vs 0.6193 unedited.
DEFAULT_N_SHOTS = 0
DEFAULT_DEV_SIZE = 160
DEV_SEED = 42


def provider_name() -> str:
    return os.getenv("LLM_PROVIDER", "groq").strip().lower()


def model_name(provider: str | None = None) -> str:
    override = os.getenv("LLM_MODEL", "").strip()
    if override:
        return override
    return PROVIDERS[provider or provider_name()]["default_model"]


def have_key(provider: str | None = None) -> bool:
    name = provider or provider_name()
    env = PROVIDERS[name]["key_env"]
    return bool(os.getenv(env, "").strip())

In [ ]:
%%writefile src/natoe/evaluate.py
"""Shared evaluation helpers: dev split, scoring, error analysis.

Both the CLI scripts and the notebook call these, so a number reported in the
notebook is produced by exactly the code that produced the submission.
"""

from __future__ import annotations

import time
from pathlib import Path

import pandas as pd

from .config import (DEFAULT_DEV_SIZE, DEFAULT_GUARD_LEVEL, DEFAULT_N_SHOTS,
                     DEV_SEED)
from .pipeline import Retriever, build_shots, run_dataset
from .res_scorer import (res_dataset, split_sections, parse_fields, normalize,
                         tokenize)


# --------------------------------------------------------------------- splits

def dev_split(train: pd.DataFrame, n_dev: int = DEFAULT_DEV_SIZE,
              seed: int = DEV_SEED) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Hold out `n_dev` rows for scoring; the rest becomes the few-shot pool.

    Few-shot exemplars must never include the row being scored, otherwise the
    dev number is meaningless.
    """
    shuffled = list(pd.Series(train.index).sample(frac=1.0, random_state=seed))
    dev_idx = set(shuffled[:n_dev])
    dev = train.loc[sorted(dev_idx)].reset_index(drop=True)
    pool = train.loc[[i for i in train.index if i not in dev_idx]].reset_index(drop=True)
    return dev, pool


def retrieval_text(frame: pd.DataFrame) -> pd.Series:
    return (frame.body_part + " " + frame.modality + " " + frame.study_description
            + " " + frame.template_content + " " + frame.dictation)


def build_retriever(pool: pd.DataFrame) -> Retriever:
    return Retriever(retrieval_text(pool).tolist())


# ------------------------------------------------------------------ scoring

def score_predictions(frame: pd.DataFrame, preds) -> dict:
    return res_dataset(frame["report"], pd.Series(list(preds), index=frame.index),
                       frame["template_content"])


def baselines(frame: pd.DataFrame) -> pd.DataFrame:
    """Dumb reference points, always worth quoting alongside a real run."""
    rows = [("ORACLE: the reference report", score_predictions(frame, frame.report)["RES"]),
            ("B1: template submitted unedited",
             score_predictions(frame, frame.template_content)["RES"])]

    raw = []
    for _, r in frame.iterrows():
        fb, _, _ = split_sections(r.template_content)
        raw.append(f"FINDINGS:\n{fb.strip()}\n\nIMPRESSION:\n{r.dictation.strip()}")
    rows.append(("B2: template FINDINGS + raw dictation as IMPRESSION",
                 score_predictions(frame, raw)["RES"]))
    return pd.DataFrame(rows, columns=["submission", "RES"]).sort_values("RES")


# ------------------------------------------------------------------ analysis

def per_case(res: dict, frame: pd.DataFrame) -> pd.Series:
    return pd.Series([x["RES"] for x in res["rows"]], index=frame.index)


def field_loss_table(res: dict) -> pd.DataFrame:
    """Where the FINDINGS loss actually is, aggregated by field label."""
    agg: dict[str, dict] = {}
    for case in res["rows"]:
        for label, d in case["fields"].items():
            if d["edit"] <= 0.02:
                continue
            slot = agg.setdefault(d.get("ref_display", label),
                                  {"weighted_loss": 0.0, "missing": 0, "cases": 0})
            slot["weighted_loss"] += d["w"] * d["edit"]
            slot["missing"] += int(bool(d.get("missing")))
            slot["cases"] += 1
    if not agg:
        return pd.DataFrame(columns=["weighted_loss", "share", "missing", "cases"])
    t = pd.DataFrame(agg).T
    t["share"] = t.weighted_loss / t.weighted_loss.sum()
    return t.sort_values("weighted_loss", ascending=False)


def diagnosis_table(train: pd.DataFrame) -> pd.DataFrame:
    """The style statistics that justify the prompt and the renderer."""
    rows = []
    for _, r in train.iterrows():
        ref_f, _, _ = parse_fields(split_sections(r.report)[0])
        tpl_f, _, _ = parse_fields(split_sections(r.template_content)[0])
        changed = [k for k, v in ref_f.items()
                   if normalize(v).strip() != normalize(tpl_f.get(k, "")).strip()]
        ref_im = split_sections(r.report)[1]
        overlap = len(set(tokenize(ref_im)) & set(tokenize(r.dictation))) / \
            max(1, len(set(tokenize(ref_im))))
        rows.append({
            "n_fields": len(ref_f),
            "n_changed": len(changed),
            "frac_changed": len(changed) / max(1, len(ref_f)),
            "impression_overlap_with_dictation": overlap,
            "impression_equals_template":
                ref_im.strip() == split_sections(r.template_content)[1].strip(),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------- runner

def run_and_score(llm, frame: pd.DataFrame, retriever: Retriever,
                  pool: pd.DataFrame, n_shots: int = DEFAULT_N_SHOTS,
                  guard_level: str = DEFAULT_GUARD_LEVEL,
                  cache_path: str | Path | None = None,
                  progress: bool = True) -> tuple[list[str], dict]:
    t0 = time.time()
    preds = run_dataset(llm, frame, retriever, pool, n_shots=n_shots,
                        guard_level=guard_level,
                        cache_path=str(cache_path) if cache_path else None,
                        progress=progress)
    res = score_predictions(frame, preds)
    res["seconds"] = time.time() - t0
    return preds, res


# ---------------------------------------------------------------- submission

def validate_submission(sub: pd.DataFrame, test: pd.DataFrame) -> list[str]:
    """Structural checks a submission must pass. Empty list means valid.

    Catches the failure modes that would silently cost leaderboard score or
    break the hiring review: wrong columns, a mismatched case_id set, a
    missing section, leaked dictation text, unlabelled FINDINGS content, and
    any field label the template did not contain.
    """
    errs: list[str] = []
    if list(sub.columns) != ["case_id", "report"]:
        errs.append(f"columns must be exactly case_id,report; "
                    f"got {list(sub.columns)}")
    if len(sub) != len(test):
        errs.append(f"row count {len(sub)} != test rows {len(test)}")
    if sub.case_id.duplicated().any():
        errs.append("duplicate case_id")
    if set(sub.case_id) != set(test.case_id):
        errs.append("case_id set does not match test.csv")

    tpl_by_id = test.set_index("case_id").template_content
    for cid, rep in sub.set_index("case_id").report.items():
        if not isinstance(rep, str) or not rep.strip():
            errs.append(f"{cid}: empty report")
            continue
        if "FINDINGS:" not in rep:
            errs.append(f"{cid}: missing FINDINGS:")
        if "IMPRESSION:" not in rep:
            errs.append(f"{cid}: missing IMPRESSION:")
        if rep.index("FINDINGS:") > rep.index("IMPRESSION:"):
            errs.append(f"{cid}: IMPRESSION before FINDINGS")
        f, _, unlab = parse_fields(split_sections(rep)[0])
        if unlab:
            errs.append(f"{cid}: unlabelled FINDINGS text {unlab[:50]!r}")
        tpl_f, _, _ = parse_fields(split_sections(tpl_by_id[cid])[0])
        invented = set(f) - set(tpl_f)
        if invented:
            errs.append(f"{cid}: labels absent from the template: "
                        f"{sorted(invented)}")
    return errs

In [ ]:
import importlib, sys
from pathlib import Path

sys.path.insert(0, "src")
for m in ("natoe", "natoe.res_scorer", "natoe.config",
          "natoe.pipeline", "natoe.evaluate"):
    if m in sys.modules:
        importlib.reload(sys.modules[m])
    else:
        importlib.import_module(m)

import natoe
from natoe import config as C
from natoe.res_scorer import (res_dataset, res_case, split_sections,
                              parse_fields, normalize, tokenize,
                              weighted_word_edit)
from natoe.pipeline import (LLM, MockLLM, Retriever, build_shots,
                            build_user_prompt, gold_patch, guard, render,
                            parse_json, gold_json, run_dataset)
from natoe.evaluate import (dev_split, build_retriever, baselines,
                            score_predictions, run_and_score, per_case,
                            field_loss_table, diagnosis_table)
print("natoe", natoe.__version__, "| provider:", C.provider_name(),
      "| model:", C.model_name())

## 3. The data

In [ ]:
TRAIN = pd.read_csv(C.TRAIN_CSV)
TEST  = pd.read_csv(C.TEST_CSV)
SAMPLE = pd.read_csv(C.SAMPLE_SUBMISSION_CSV)

print("train", TRAIN.shape, "| test", TEST.shape, "| sample", SAMPLE.shape)
print("train cols:", list(TRAIN.columns))
print("test  cols:", list(TEST.columns))
display(TRAIN["modality"].value_counts().to_frame("train")
        .join(TEST["modality"].value_counts().to_frame("test"))
        .fillna(0).astype(int))

### One worked example, so the shape of the data is concrete

In [ ]:
_ex = TRAIN[(TRAIN.body_part == "Chest") & (TRAIN.modality == "XRAY")].iloc[0]
print(f"STUDY: {_ex.study_description} | age {_ex.patient_age_band} | {_ex.patient_sex}")
print("\n--- DICTATION ---\n" + _ex.dictation)
print("\n--- TEMPLATE (the normal report we must edit) ---\n" + _ex.template_content)
print("\n--- REFERENCE REPORT ---\n" + _ex.report)

Note the house style: the dictated abnormality (`Mild thoracic spondylosis`) is
**prepended** to the still-true template sentence inside `BONES`, the untouched
fields are copied verbatim, and the IMPRESSION became a numbered list. This is
not guessable from the rules alone — which is why few-shot exemplars are
retrieved from `train.csv` rather than hand-written.

## 4. The metric, re-implemented

`RES_case = 0.65·F + 0.35·I`

* **`I`** — weighted word edit distance over the whole IMPRESSION section.
* **`F`** — field-aware FINDINGS score. For each reference field, first ask
  *did this field change from the template?* changed → weight **3**,
  unchanged → weight **1**. Then `F = Σ(w × word_edit) / Σ(w)`.

Token weights: clinically critical words / numbers / units = **4.0**, other
content words = **2.0**, function words = **0.25**. Each field's cost is divided
by the larger total token weight of reference vs submission, so it lands in
`[0, 1]`.

**What it rewards**

1. Leaving normal fields untouched — weight 1 *and* free to match exactly.
2. Copying the radiologist's words instead of paraphrasing — a synonym is a
   weight-2 substitution; a changed measurement is weight-4.
3. Correct field **routing** — a finding under the wrong label counts as
   missing from the right field *and* as extra content under the wrong one.
4. Not inventing findings. Unsupported content is pure penalty.

> The exact private scorer may differ in the unspecified corners (the
> function-word list, how "unexpected field" penalties are quantified). Use
> this for **relative** comparison between variants.

In [ ]:
_ref = """FINDINGS:
LUNGS: Mild right basilar airspace opacity. No pulmonary edema.
PLEURA: Small right pleural effusion. No pneumothorax.
OSSEOUS STRUCTURES: No acute osseous abnormality identified on these views.

IMPRESSION:
Mild right basilar airspace opacity and small right pleural effusion."""

_tpl = """FINDINGS:
LUNGS: No focal airspace opacity or pulmonary edema.
PLEURA: No pleural effusion or pneumothorax.
OSSEOUS STRUCTURES: No acute osseous abnormality identified on these views.

IMPRESSION:
No acute cardiopulmonary abnormality."""

for name, sub in [
    ("perfect submission", _ref),
    ("template left unedited", _tpl),
    ("effusion filed under LUNGS",
     _ref.replace("PLEURA: Small right pleural effusion. No pneumothorax.",
                  "LUNGS: Small right pleural effusion. No pneumothorax.")),
]:
    r = res_case(_ref, sub, _tpl)
    print(f"{name:28s} RES={r['RES']:.3f}  (F={r['F']:.3f}, I={r['I']:.3f})")

print()
print("air-space == airspace :", tokenize("air-space") == tokenize("airspace"))
print("8 mm == 8 millimetres :",
      weighted_word_edit("8 mm nodule", "8 millimetres nodule") == 0.0)
print("L5-S1 == l5s1         :", tokenize("L5-S1") == tokenize("l5s1"))

## 5. Baselines

Measure the dumb options before trying to be clever.

In [ ]:
print("ON THE FULL TRAIN SET")
print(baselines(TRAIN).to_string(index=False))
print("\n'B1' is the number every real submission has to beat.")

## 6. What the reference reports actually look like

These measurements determined the design of the prompt and the renderer.

In [ ]:
diag = diagnosis_table(TRAIN)
print(f"fields per reference report          : {diag.n_fields.mean():.2f}")
print(f"fields that DIFFER from the template : {diag.n_changed.mean():.2f}"
      f"  ({100 * diag.frac_changed.mean():.0f}% of all fields)")
print(f"reports with zero changed fields     : {(diag.n_changed == 0).sum()}")
print(f"IMPRESSION >80% dictation vocabulary : "
      f"{(diag.impression_overlap_with_dictation > 0.8).sum()} / {len(diag)}")
print(f"IMPRESSION 40-80%                    : "
      f"{((diag.impression_overlap_with_dictation > 0.4) & (diag.impression_overlap_with_dictation <= 0.8)).sum()}")
print(f"IMPRESSION identical to the template : "
      f"{diag.impression_equals_template.sum()} / {len(diag)}")

In [ ]:
# Where does a CHANGED field's wording come from: the dictation or the template?
from collections import Counter
src = Counter()
for _, r in TRAIN.iterrows():
    ref_f, _, _ = parse_fields(split_sections(r.report)[0])
    tpl_f, _, _ = parse_fields(split_sections(r.template_content)[0])
    d_tok, t_tok = set(tokenize(r.dictation)), set(tokenize(r.template_content))
    for k, v in ref_f.items():
        if normalize(v).strip() == normalize(tpl_f.get(k, "")).strip():
            continue
        v_tok = set(tokenize(v))
        nd, nt = len(v_tok & d_tok), len(v_tok & t_tok)
        src["from_dictation" if nd > nt else
            ("from_template" if nt > nd else "mixed")] += 1
tot = sum(src.values())
for k, v in src.most_common():
    print(f"  {k:15s} {v:>5}  ({100 * v / tot:.0f}%)")

### Conclusions that shape the design

1. **~63% of fields change.** A "minimal edit" instinct is wrong. The
   instruction is *change every field the dictation speaks to, and nothing
   else* — not *touch as little as possible*.
2. **Changed fields copy the radiologist's words** (76% reuse dictation
   vocabulary). So: **copy verbatim, never paraphrase.**
3. **IMPRESSION is lifted from the dictation** in 60% of cases. The dictation
   usually already contains the radiologist's own summary; we should be
   *selecting* it, not writing it.
4. **Field order follows the template.** The renderer must always emit the
   template's labels in the template's order.

## 7. Does the architecture even work? (ceiling test)

Before spending a single API call: feed the renderer the **gold** patch — the
exact per-field edits the reference made — and score it. If that is not ≈0,
the assembly is broken and no amount of prompting will save us.

In [ ]:
rows = []
for level in ("off", "numbers", "strict"):
    outs = []
    for _, r in TRAIN.iterrows():
        case = {"template_content": r.template_content, "dictation": r.dictation}
        outs.append(render(case, guard(gold_patch(r), case, level)))
    res = score_predictions(TRAIN, outs)
    rows.append({"guard level": level, "RES": res["RES"],
                 "F": res["F"], "I": res["I"]})
display(pd.DataFrame(rows))

Reading:

* **`off` → 0.0023.** The renderer reproduces the reference almost exactly, so
  field order, label spelling and untouched-field copying are all correct. The
  architecture is sound and the LLM only has to get the *content* right.
* **`numbers` → 0.0030.** Dropping sentences that introduce a measurement found
  in neither the dictation nor the template costs 0.001. Cheap insurance.
* **`strict` → 0.0154.** Also policing laterality/severity starts deleting text
  the reference legitimately contains.

=> ship **`guard="numbers"`**.

An earlier version of this guard also policed laterality and severity and cost
**0.19 RES**: the reference reports themselves add laterality the dictation
never stated, so "unsupported laterality" is not a safe thing to strip.
Negation is never a trigger either — templates legitimately say
"No pneumothorax." when the dictation never mentioned pneumothorax.

## 8. Few-shot exemplars, retrieved

`train.csv` is the style manual. For each case we retrieve the most similar
training rows (body part + modality + template + dictation) and show the model
their template → dictation → **gold patch** triple.

Similarity is TF-IDF cosine over normalised tokens, implemented in
`pipeline.Retriever` so the notebook has no extra dependency.

In [ ]:
DEV, POOL = dev_split(TRAIN, n_dev=C.DEFAULT_DEV_SIZE)
print(f"dev {len(DEV)} rows (scored) | few-shot pool {len(POOL)} rows")
print("exemplars never include the row being scored, so the dev number is honest.")

RETRIEVER = build_retriever(POOL)
_probe = dict(DEV.iloc[0])
_probe_query = (f"{_probe['body_part']} {_probe['modality']} "
                f"{_probe['study_description']} {_probe['template_content']} "
                f"{_probe['dictation']}")
print(f"\nprobe case: {_probe['body_part']} / {_probe['modality']} | "
      f"{_probe['study_description']}")
for i in RETRIEVER.query(_probe_query, top_k=3):
    print(f"  -> {POOL.body_part[i]:14s} {POOL.modality[i]:5s} | "
          f"{str(POOL.dictation[i])[:70]!r}")

In [ ]:
_shots = build_shots(RETRIEVER, POOL, _probe, n_shots=1)
_prompt = build_user_prompt(_probe, _shots)
print(f"prompt with 1 shot: ~{len(_prompt) // 4} tokens\n")
print(_prompt[-2200:])

## 9. The model

Both providers speak the OpenAI chat API, so one client covers both and
switching is a one-line change. Free tiers are enough — 132 test rows.

| provider | endpoint | default model |
|---|---|---|
| `groq` | `api.groq.com/openai/v1` | `openai/gpt-oss-120b` |
| `openrouter` | `openrouter.ai/api/v1` | `google/gemma-4-31b-it:free` |

Two operational details that mattered a lot in practice:

* **`reasoning_effort=low`** — reasoning models spend most of their output
  budget on chain-of-thought. This task does not need it, and it is a ~4×
  speed difference (13.6s → 3.3s per call).
* **low concurrency** — free tiers rate-limit hard. `workers=4` with
  exponential backoff; failures are surfaced, never cached.

In [ ]:
if C.have_key():
    LLM_RUNNER = LLM(provider=C.provider_name(), model=os.getenv("LLM_MODEL") or None)
    print("provider:", LLM_RUNNER.provider, "| model:", LLM_RUNNER.model)
    print("chain   :", LLM_RUNNER.chain)
else:
    LLM_RUNNER = MockLLM()
    print("No API key found -> MockLLM.")
    print("The plumbing runs, but the score below is MEANINGLESS.")
    print("Add GROQ_API_KEY or OPENROUTER_API_KEY to .env and re-run.")

In [ ]:
# one row, end to end, so the contract is visible
_row = dict(DEV.iloc[1])
_case = {"template_content": _row.template_content, "dictation": _row.dictation,
         "modality": _row.modality, "body_part": _row.body_part,
         "study_description": _row.study_description,
         "patient_age_band": _row.patient_age_band,
         "patient_sex": _row.patient_sex}
_raw = LLM_RUNNER.complete(build_user_prompt(_case, build_shots(RETRIEVER, POOL, _case, 2)))
print("RAW MODEL OUTPUT (truncated):\n" + _raw[:600])
print("\n--- RENDERED REPORT ---\n" + render(_case, guard(parse_json(_raw), _case, "numbers")))
print("\n--- REFERENCE ---\n" + _row.report)

## 10. Run on the dev split and score

In [ ]:
dev_preds, dev_res = run_and_score(LLM_RUNNER, DEV, RETRIEVER, POOL,
                                   n_shots=C.DEFAULT_N_SHOTS,
                                   guard_level=C.DEFAULT_GUARD_LEVEL,
                                   cache_path=C.CACHE_PATH)
print(f"\nDEV RES = {dev_res['RES']:.4f}   F = {dev_res['F']:.4f}   "
      f"I = {dev_res['I']:.4f}   ({dev_res['seconds']:.0f}s)")

_b = baselines(DEV)
print("\n" + _b.to_string(index=False))
print(f"\n" + pd.DataFrame([{
    "submission": f"pipeline ({C.DEFAULT_N_SHOTS} shots, "
                  f"guard={C.DEFAULT_GUARD_LEVEL})",
    "RES": dev_res["RES"]}]).to_string(index=False))

The headline number to watch is **how much of the gap between "template
unedited" and the oracle we closed**. That, not the absolute RES, is what tells
us whether a change helped.

In [ ]:
_b = baselines(DEV)
base, oracle = float(_b.iloc[1]["RES"]), 0.0
print(f"template unedited : {base:.4f}")
print(f"oracle            : {oracle:.4f}")
print(f"pipeline          : {dev_res['RES']:.4f}")
print(f"\nclosed {100 * (base - dev_res['RES']) / max(1e-9, base - oracle):.0f}% "
      f"of the gap to the oracle")

## 11. Error analysis

RES is an average, so the mean hides which cases fail. The worst cases are
where the remaining work is.

In [ ]:
per = per_case(dev_res, DEV)
print(per.describe(percentiles=[.25, .5, .75, .9, .95]).round(3).to_string())

In [ ]:
loss = field_loss_table(dev_res)
display(loss.head(15).round(4))
if len(loss):
    print(f"top 10 field labels = {loss.share.head(10).sum():.0%} of total FINDINGS loss")

In [ ]:
for i in per.sort_values(ascending=False).index[:3]:
    r = DEV.loc[i]
    print("=" * 96)
    print(f"RES={per[i]:.3f}  {r.body_part}/{r.modality}  {r.study_description}"
          f"  age {r.patient_age_band} {r.patient_sex}")
    print("--- DICTATION ---\n" + str(r.dictation)[:420])
    print("--- PREDICTION ---\n" + dev_preds[i][:620])
    print("--- REFERENCE ---\n" + str(r.report)[:620])

## 12. Iterate

Three knobs. All cheap, because responses are cached on disk by
`(model, guard_level, n_shots, prompt)` — only genuinely new prompts cost a call.

| knob | trades |
|---|---|
| `n_shots` 0 → 1 → 3 | prompt tokens vs. house-style fidelity |
| `guard_level` off / numbers / strict | measured in §7; `numbers` is nearly free |
| provider / model | instruction-following vs. cost and rate limits |

The knob that usually matters most is the **prompt**: if error analysis shows
routing mistakes, the fix belongs in the rules of the system prompt, not in a
bigger model.

In [ ]:
variants = []
for n_shots in (0, 1, 3):
    for level in ("numbers", "strict"):
        _, res = run_and_score(LLM_RUNNER, DEV, RETRIEVER, POOL,
                               n_shots=n_shots, guard_level=level,
                               cache_path=C.CACHE_PATH, progress=False)
        variants.append({"n_shots": n_shots, "guard": level,
                         "RES": res["RES"], "F": res["F"], "I": res["I"],
                         "s": round(res["seconds"])})
        print(f"n_shots={n_shots} guard={level:8s} -> RES={res['RES']:.4f} "
              f"F={res['F']:.4f} I={res['I']:.4f} ({res['seconds']:.0f}s)")

VARIANTS = pd.DataFrame(variants).sort_values("RES")
display(VARIANTS)
BEST = VARIANTS.iloc[0]
print(f"\nbest: n_shots={int(BEST.n_shots)} guard={BEST.guard}")

## 13. Predict the test set and write `submission.csv`

For the real test set *every* train row is legitimately available as a
few-shot exemplar, including the ones held out during dev evaluation.

In [ ]:
N_SHOTS, LEVEL = int(BEST.n_shots), BEST.guard
TEST_RETRIEVER = build_retriever(TRAIN)      # full train set as the pool

test_preds, _ = run_and_score(LLM_RUNNER, TEST, TEST_RETRIEVER, TRAIN,
                              n_shots=N_SHOTS, guard_level=LEVEL,
                              cache_path=C.CACHE_PATH)

SUB = pd.DataFrame({"case_id": TEST.case_id, "report": test_preds})
import csv as _csv
OUT_PATH = C.OUTPUT_DIR / "submission.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SUB.to_csv(OUT_PATH, index=False, quoting=_csv.QUOTE_ALL)
print(f"wrote {OUT_PATH}  {SUB.shape}   (n_shots={N_SHOTS}, guard={LEVEL})")

In [ ]:
import csv

def validate(sub, test):
    errs = []
    if list(sub.columns) != ["case_id", "report"]:
        errs.append(f"columns must be exactly case_id,report; got {list(sub.columns)}")
    if len(sub) != len(test):
        errs.append(f"row count {len(sub)} != {len(test)}")
    if sub.case_id.duplicated().any():
        errs.append("duplicate case_id")
    if set(sub.case_id) != set(test.case_id):
        errs.append("case_id set does not match test.csv")

    tpl_by_id = test.set_index("case_id").template_content
    for cid, rep in sub.set_index("case_id").report.items():
        if not isinstance(rep, str) or not rep.strip():
            errs.append(f"{cid}: empty report"); continue
        if "FINDINGS:" not in rep:   errs.append(f"{cid}: missing FINDINGS:")
        if "IMPRESSION:" not in rep: errs.append(f"{cid}: missing IMPRESSION:")
        if rep.index("FINDINGS:") > rep.index("IMPRESSION:"):
            errs.append(f"{cid}: IMPRESSION before FINDINGS")
        f, _, unlab = parse_fields(split_sections(rep)[0])
        if unlab:
            errs.append(f"{cid}: unlabelled FINDINGS text {unlab[:50]!r}")
        tpl_f, _, _ = parse_fields(split_sections(tpl_by_id[cid])[0])
        if set(f) - set(tpl_f):
            errs.append(f"{cid}: labels absent from the template: "
                        f"{sorted(set(f) - set(tpl_f))}")
    return errs

errs = validate(SUB, TEST)
print("PASS - submission is structurally valid" if not errs
      else f"{len(errs)} PROBLEM(S):")
for e in errs[:20]:
    print("  -", e)

print(f"\nmean report length: {SUB.report.str.len().mean():.0f} chars")
print("\n--- first prediction ---\n" + SUB.report.iloc[0])

In [ ]:
print("Submit:\n"
      "  kaggle competitions submit -c radiology-reporting-harness \\\n"
      f"      -f {OUT_PATH} -m \"template-edit pipeline: LLM JSON patch + deterministic render\"\n\n"
      "With the pipeline notebook attached (paste the notebook URL into the\n"
      "submission description on the Kaggle page):\n"
      "  kaggle competitions submit -c radiology-reporting-harness \\\n"
      f"      -f {OUT_PATH} -k kamalnayan90/<NOTEBOOK> -v <VERSION> -m \"...\"")

## 14. What is left on the table, and the fine-tuning path

The prompt pipeline is the right **first** system: cheap, auditable, and every
failure has a named cause. But the metric rewards verbatim copying, and no
amount of prompting fully removes paraphrasing. That is the natural place for
stage 2.

**Stage 2 — synthetic-data alignment**

1. From every train row, *derive* a dictation from the reference report (or
   forward-generate one), producing `(dictation, template, report)` triples.
2. LoRA fine-tune a small open model (e.g. Qwen 2.5 7B/14B) on the JSON-patch
   format. The output space is tiny and highly templated, so a 7B model learns
   it quickly and inference gets far cheaper than the API.
3. Keep the **identical** guard and render code, so the fine-tuned model is a
   drop-in swap and the guardrails still apply.

**Where Jev fits**

Jev (`typesafe/jev-1.13`, ~$0.042/M input, output free) is a *decision* model,
not a text generator — it returns calibrated probabilities for questions you
define. That suits this task precisely, because two of the remaining decisions
are not generation problems at all:

* *"does this dictated finding belong in field X?"* — a `choice` question over
  the template's field labels. This is the routing error we keep seeing.
* *"is this generated sentence supported by the dictation?"* — a `noul`
  question, which is exactly what the current regex guard approximates.

So the natural end state is a **generative model for the wording, Jev for the
decisions**, with the same renderer in front.

**Design decisions worth defending in review**

* *The model never writes the report.* It writes a patch; Python renders. This
  makes field order, label spelling and untouched-field preservation
  structurally impossible to get wrong.
* *The guard is deliberately narrow.* Unsupported **measurements** are the
  defensible thing to strip (cost 0.001). Unsupported laterality is not, because
  the reference itself adds it (cost 0.19).
* *No clinical inference from the context columns.* `modality`, `body_part`,
  `study_description`, `patient_age_band`, `patient_sex` are used only to select
  few-shot exemplars, never as a source of findings.
* *Reproducible.* Temperature 0, on-disk response cache, a single `.env`, and a
  submission validator that must pass before the file is handed over.